# pandas와 연구 데이터셋

## OWID CO₂ 자료의 구조와 품질 점검

Python 코드를 통해 데이터를 가공하기 
국가–연도 환경자료의 관측 단위, 식별자, 요약통계, 결측과 중복, 변수의 단위와 범위 확인

데이터셋 내, 컬럼별 의미와 품질을 검사하는 방법을 확인하고 보고서를 작성


## 사용 데이터

### 강의: OWID CO₂ 수업용 자료

| 항목 | 내용 |
|---|---|
| 출처 | Our World in Data CO₂ 공개자료에서 수업 범위만 추출한 snapshot |
| 규모 | 175행, 5개 지역, 1990~2024년 |
| 한 행의 의미 | 한 지역의 한 연도 CO₂ 배출 관측 |
| 주요 열 | `entity`, `code`, `year`, `emissions_total`, `emissions_total_per_capita` |
| 단위 | 총배출량은 tonnes, 1인당 배출량은 tonnes/person |


### 실습: UCI Bike Sharing

| 항목 | 내용 |
|---|---|
| 출처 | UCI Machine Learning Repository의 Bike Sharing Dataset |
| 규모 | 17,379행, 2011~2012년 시간별 기록 |
| 한 행의 의미 | 특정 날짜의 특정 시간대에 기록된 대여수요와 날씨·달력 조건 |
| 주요 열 | `dteday`, `hr`, `season`, `workingday`, `weathersit`, `temp`, `hum`, `casual`, `registered`, `cnt` |

`season`, `weekday`, `weathersit`는 숫자로 저장된 범주 / 카테고리컬 데이터 

`cnt = casual + registered`



## 이번 교시의 학습목표

수업이 끝나면 다음을 수행할 수 있습니다.

- 변수 할당, 함수 호출, 열 선택이 포함된 짧은 pandas 코드를 실행 순서에 맞게 읽습니다.
- `DataFrame`, `Series`, `Index`와 실제 연구자료의 관측 단위(`grain`)·식별자(`key`)를 구분합니다.
- 평균, 중앙값, 사분위수가 수치형 변수의 분포를 서로 다른 관점에서 요약한다는 점을 설명합니다.
- `head`, `info`, `dtypes`, `isna`, `duplicated`, `loc`를 이용해 자료형·결측·중복·값의 범위를 점검합니다.
- 총배출량과 1인당 배출량, 국가와 집계지역을 구분해 분석에 사용할 행과 열을 명시합니다.


## pandas로 연구자료를 다룬다는 것

**Python**은 분석 절차를 작성하고 실행하는 언어이며, **pandas**는 행과 열로 이루어진 자료를 다루는 대표적인 Python 라이브러리입니다.

연구에서 pandas를 사용하는 목적은 자료를 어떻게 읽고, 어떤 관측을 남기고, 무엇을 계산했는지 다음 사람이 다시 확인할 수 있도록 분석 절차를 코드로 기록하는 데 있습니다.

`자료 읽기 → 구조 파악 → 행·열 선택 → 요약 → 품질 점검 → 분석용 표 생성`

이번 교시에서는 함수 이름을 외우기보다 **한 행의 의미, 변수의 형태, 요약값의 의미**를 출력 결과와 연결합니다.


## DataFrame·Series·Index를 하나의 표에서 구분하기

| pandas 객체 | 화면에서 보이는 형태 | 연구자료에서의 역할 |
|---|---|---|
| `DataFrame` | 여러 행과 여러 열로 이루어진 표 | 관측과 변수를 함께 보관 |
| `Series` | 표에서 선택한 열 하나 | 한 변수의 값·자료형·결측 확인 |
| `Index` | 표 왼쪽의 행 표식 | pandas가 행을 선택하고 정렬할 때 사용하는 label |
| `key` | 실제 열 하나 또는 여러 열의 조합 | 현실의 관측을 고유하게 식별 |

> 기본 `Index`가 0, 1, 2로 고유하더라도 실제 관측을 나타내는 `국가 + 연도` 조합이 중복될 수 있습니다. 따라서 Index와 `key`를 같은 개념으로 보지 않습니다.

다음 실행 셀에서는 표 전체, 열 하나, Index, `key` 후보를 같은 자료에서 나란히 확인합니다.


In [ ]:
import pandas as pd
from IPython.display import display

observations = pd.DataFrame(
    {
        "code": ["KOR", "JPN", "USA", "KOR"],
        "year": [2021, 2021, 2021, 2022],
        "co2_mt": [615.0, 1080.0, 5010.0, 602.0],
    }
)
one_variable = observations["co2_mt"]

print("[DataFrame] 여러 변수와 관측을 함께 가진 표")
display(observations)
print(f"shape = {observations.shape}  →  {observations.shape[0]}행 × {observations.shape[1]}열")

print("\n[Series] co2_mt 열 하나")
display(one_variable)

object_summary = pd.DataFrame(
    {
        "확인 대상": ["DataFrame", "Series", "Index", "`key` 후보"],
        "실제 결과": [
            type(observations).__name__,
            type(one_variable).__name__,
            observations.index.tolist(),
            "code + year",
        ],
    }
)
print("\n[pandas 객체와 연구자료 개념의 연결]")
display(object_summary)
print("Index는 행 표식이고, code + year는 현실의 관측을 찾는 key 후보입니다.")


## grain과 key로 한 행의 의미를 확인하기

| 용어 | 질문 | 국가별 CO₂ 자료의 예 | 시간별 대여자료의 예 |
|---|---|---|---|
| grain | 한 행은 현실에서 무엇인가? | 한 국가의 한 연도 | 한 날짜의 한 시간대 |
| key | 어떤 열 조합이 그 행을 고유하게 찾는가? | `code + year` | `date + hour` |
| measure | 무엇을 측정하거나 계산했는가? | CO₂ 배출량 | 대여건수 |
| dimension | 어떤 조건으로 관측을 나누는가? | 국가, 연도 | 요일, 날씨, 시간 |

grain은 분석자가 표를 읽는 기준이며, key는 그 기준이 실제 데이터에서 지켜지는지 검사하는 수단입니다. key 후보는 열 이름만 보고 확정하지 않고 `duplicated`로 검증합니다.

다음 실행 셀에서는 고의로 중복된 국가–연도 행을 찾아 grain과 key가 일치하는지 확인합니다.


In [ ]:
import pandas as pd
from IPython.display import display

country_year = pd.DataFrame(
    {
        "code": ["KOR", "KOR", "JPN", "JPN", "KOR"],
        "year": [2021, 2022, 2021, 2022, 2022],
        "co2_mt": [615.0, 602.0, 1080.0, 1055.0, 603.5],
        "source": ["A", "A", "A", "A", "B"],
    }
)
key_columns = ["code", "year"]
duplicate_mask = country_year.duplicated(key_columns, keep=False)

print("[원자료] 한 행을 '한 국가의 한 연도'라고 가정합니다.")
display(country_year)
print(f"key 후보: {key_columns}")
print(f"중복된 key에 속한 행 수: {int(duplicate_mask.sum())}")

print("\n[중복 key 행]")
display(country_year.loc[duplicate_mask].sort_values(key_columns + ["source"]))

key_audit = pd.DataFrame(
    {
        "점검": ["전체 행", "고유한 code-year", "중복 key 행"],
        "결과": [
            len(country_year),
            country_year[key_columns].drop_duplicates().shape[0],
            int(duplicate_mask.sum()),
        ],
    }
)
display(key_audit)
print("결론: code + year가 중복되므로 출처 차이를 확인하거나 중복 처리 규칙을 정해야 합니다.")


## 열을 선택하고 자료형을 확인하는 이유

pandas의 자료형은 어떤 계산이 가능한지를 결정합니다.

| 자료형의 예 | 주로 담는 값 | 먼저 확인할 문제 |
|---|---|---|
| 정수·실수 | 배출량, 온도, 대여건수 | 단위, 결측, 비정상 범위 |
| 문자열·범주 | 국가코드, 부하유형, 날씨등급 | 표기 불일치, 범주 누락 |
| 날짜시간 | 측정시각, 연도·월·시간 | 파싱 실패, 시간대, 정렬 순서 |

숫자로 저장됐더라도 `1=맑음, 2=흐림`처럼 범주를 뜻할 수 있고, 쉼표가 포함된 숫자가 문자열로 읽힐 수도 있습니다. 따라서 `dtypes`를 본 뒤 변수 설명과 실제 값의 예를 함께 확인합니다.


## 평균·중앙값·사분위수·표준편차를 함께 읽기

기술통계는 자료를 하나의 숫자로 단순화하는 작업이 아니라, 분포의 중심과 퍼짐을 빠르게 점검하는 과정입니다.

| 값 | 답하는 질문 | 주의할 점 |
|---|---|---|
| 평균 | 전체 값을 고르게 나눴을 때 중심은 어디인가? | 큰 값과 작은 값의 영향을 많이 받음 |
| 중앙값 | 값을 정렬했을 때 가운데는 어디인가? | 극단값에 비교적 덜 민감함 |
| 사분위수 | 가운데 50%는 어느 구간에 모이는가? | 분포의 비대칭을 함께 확인해야 함 |
| 표준편차 | 값이 평균 주변에서 얼마나 퍼지는가? | 원래 변수와 같은 단위지만 극단값의 영향을 받음 |

평균과 중앙값의 차이가 크다면 일부 큰 관측이 평균을 끌어올렸을 가능성을 살펴봅니다. 다음 실행 셀에서는 숫자표, 요약통계, 히스토그램과 상자그림을 한 화면에서 연결합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sample = pd.DataFrame(
    {
        "site": ["A", "B", "C", "D", "E", "F", "G", "H"],
        "pm25": [18.0, 20.0, 21.0, 22.0, 24.0, 25.0, np.nan, 75.0],
    }
)
values = sample["pm25"]
summary = values.describe().rename(
    {
        "count": "유효 관측 수",
        "mean": "평균",
        "std": "표준편차",
        "min": "최솟값",
        "25%": "1사분위수",
        "50%": "중앙값",
        "75%": "3사분위수",
        "max": "최댓값",
    }
).to_frame("PM2.5")

print("[작은 환경 측정자료]")
display(sample)
print("\n[기술통계]")
display(summary.round(2))
print(f"평균 {values.mean():.1f}, 중앙값 {values.median():.1f}")
print("평균이 중앙값보다 큰 이유를 75인 관측과 연결해 확인합니다.")

valid = values.dropna()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].hist(valid, bins=[10, 20, 30, 40, 50, 60, 70, 80], color="#4C78A8", edgecolor="white")
axes[0].axvline(valid.mean(), color="#E45756", linestyle="--", label="mean")
axes[0].axvline(valid.median(), color="#54A24B", linestyle=":", label="median")
axes[0].set(title="Distribution and two centers", xlabel="PM2.5", ylabel="Rows")
axes[0].legend()
axes[1].boxplot(valid, vert=False)
axes[1].set(title="Quartiles and large observations", xlabel="PM2.5", yticks=[])
plt.tight_layout()
plt.show()


## 결측·중복·관계식을 별도의 품질 조건으로 검사하기

자료 품질은 한 번의 `PASS`로 끝나지 않습니다. 서로 다른 오류는 서로 다른 질문과 코드가 필요합니다.

1. **결측:** 필요한 값이 비어 있는가?
2. **중복:** `key`가 같은 행이 둘 이상인가?
3. **범위:** 물리적으로 또는 정의상 불가능한 값이 있는가?
4. **관계식:** 구성 열의 합이나 비율이 정의와 일치하는가?

예를 들어 총 대여건수가 회원 대여와 비회원 대여의 합으로 정의된다면 `total = member + casual`을 별도로 검사할 수 있습니다. 다음 실행 셀에서는 네 종류의 점검 결과와 문제가 있는 행을 함께 출력합니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

rentals = pd.DataFrame(
    {
        "date": ["2024-07-01", "2024-07-01", "2024-07-01", "2024-07-01", "2024-07-01"],
        "hour": [8, 9, 9, 10, 11],
        "member": [82, 65, 65, 51, 30],
        "casual": [18, 15, 15, None, 9],
        "total": [100, 80, 80, 70, 45],
    }
)
key = ["date", "hour"]
relation_ok = rentals["total"].eq(rentals["member"] + rentals["casual"])

audit = pd.DataFrame(
    {
        "품질 조건": ["결측 셀", "중복 key 행", "음수 측정값", "관계식 위반 행"],
        "발견 수": [
            int(rentals.isna().sum().sum()),
            int(rentals.duplicated(key, keep=False).sum()),
            int((rentals[["member", "casual", "total"]] < 0).sum().sum()),
            int((~relation_ok.fillna(False)).sum()),
        ],
    }
)

print("[점검할 원자료]")
display(rentals)
print("\n[품질 조건별 결과]")
display(audit)
print("\n[관계식을 확인하지 못했거나 위반한 행]")
display(rentals.loc[~relation_ok.fillna(False)])

audit_plot = audit.assign(
    plot_label=["Missing cells", "Duplicate-key rows", "Negative values", "Identity violations"]
)
ax = audit_plot.set_index("plot_label")["발견 수"].plot.bar(
    color=["#4C78A8", "#F58518", "#54A24B", "#E45756"], figsize=(8, 3.2)
)
ax.set(title="Each quality condition needs a separate check", xlabel="", ylabel="Rows or cells")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## 분석 흐름

국가–연도 CO₂ 자료를 이용해 DataFrame 구조, grain, key, 단위와 기술통계를 확인
실습시간에서는 시간별 자전거 대여자료에 같은 절차를 실습 예정

데이터 품질 보고 예시

- 한 행이 나타내는 관측 단위
- 관측을 고유하게 찾는 key 후보와 중복 검사 결과
- 핵심 변수의 단위·자료형·분포 요약
- 발견한 결측·중복·관계식 문제와 처리 계획

pandas 코드를 실행했다는 사실보다 **출력이 자료의 어떤 성질을 확인했는지 설명할 수 있는가**


## 1. 입력 파일과 실행 환경

분석 결과의 재현성을 위해 실행환경과 입력 파일을 기록합니다.

| 확인 항목 | 강의 기준 | 불일치 시 점검 |
|---|---|---|
| 작업 폴더 | 배포 패키지 최상위 폴더 | `data` 폴더 위치 |
| 입력 파일 | `data/co2/owid_co2_sample.csv` | 파일명·버전·경로 |
| 예상 구조 | 175행 × 5열 | 다른 snapshot 사용 여부 |
| 필수 열 | `entity`, `code`, `year`, 두 배출량 열 | 열 이름과 파일 스키마 |

코드 셀은 Python·pandas 버전과 실제 파일명을 출력합니다. 이어지는 분석은 이 조건이 강의 명세와 일치할 때 수행합니다.


In [ ]:
from pathlib import Path
import platform

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행하십시오."
    )

DATA_PATH = find_course_data("co2/owid_co2_sample.csv")
print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("data file:", DATA_PATH.name)


## 데이터 품질 문제를 표에서 진단하기

오류가 섞인 여섯 행을 보고 결측, 중복 key와 열 사이 관계식 위반을 먼저 찾아보세요. 다음 코드가 각 문제를 서로 다른 검사로 확인합니다.


In [ ]:
audit_example = pd.DataFrame({
    "date": ["2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01", "2026-01-01"],
    "hour": [8, 9, 9, 10, 11, 12],
    "casual": [3, 4, 4, 5, 2, 6],
    "registered": [7, 8, 8, 9, 10, 12],
    "cnt": [10, 12, 12, None, 15, 18],
})
audit_example.loc[4, "cnt"] = 99
display(audit_example)

audit_results = pd.Series({
    "missing cnt": int(audit_example["cnt"].isna().sum()),
    "duplicated date-hour key": int(audit_example.duplicated(["date", "hour"]).sum()),
    "cnt identity violations": int((
        audit_example["cnt"].notna()
        & audit_example["cnt"].ne(audit_example["casual"] + audit_example["registered"])
    ).sum()),
}, name="problem rows")
print("[품질조건별 검사 결과]")
display(audit_results.to_frame())
print("각 검사는 서로 다른 문제를 확인하므로 하나의 0건 결과로 대신할 수 없습니다.")


## 2. DataFrame의 행·열 구조

`pd.read_csv()`는 CSV 파일을 `DataFrame`으로 읽습니다. 출력에서는 표의 형태와 관측의 의미를 연결해 해석합니다.

| 출력 | CO₂ 자료에서 확인할 내용 |
|---|---|
| `shape = (175, 5)` | 175개 관측과 5개 변수 |
| 첫 번째 행 | 한 지역의 한 연도 배출 관측 |
| `entity`, `code`, `year` | 지역과 시점을 나타내는 차원 |
| 배출량 두 열 | 총배출량과 1인당 배출량 |

첫 행을 세로로 표시해 열 이름, 값, 단위를 대응시킵니다.


In [ ]:
co2 = pd.read_csv(DATA_PATH)

print("object type:", type(co2).__name__)
print("shape (rows, columns):", co2.shape)
display(co2.head(4))
print("\n[열별 자료형과 결측이 아닌 값의 수]")
co2.info()


### 출력 해석: 한 행과 한 열

표 전체를 보기 전에 첫 번째 행을 세로로 펼쳐, 각 값이 어떤 열에 속하는지 확인합니다.


In [ ]:
print("한 행의 의미: 한 지역의 한 연도 CO2 관측")
print("열 이름:", co2.columns.tolist())
display(co2.iloc[[0]].T.rename(columns={co2.index[0]: "첫 번째 행"}))


### 출력 해석: `shape`와 `head`가 말해 주는 범위

| 이 출력에서 직접 확인한 것 | 아직 확인하지 않은 것 |
|---|---|
| snapshot의 행·열 수 | 전체 원자료와 같은 범위인지 |
| 첫 몇 행의 실제 값과 열 배치 | 측정값의 정확성·대표성 |
| 설명과 파일 구조의 1차 일치 | 누락 국가·측정오차의 존재 |

`shape`와 `head`만으로 측정 정확도나 표본 대표성을 평가할 수는 없습니다. 다음 단계에서 자료형, 결측과 key 중복을 별도로 검사합니다.


## 3. Series·DataFrame·Index와 `key`

| 표현 | 반환 형식 | 확인 대상 |
|---|---|---|
| `co2["year"]` | `Series` | 변수 하나의 값과 자료형 |
| `co2[["code", "year"]]` | `DataFrame` | 여러 열로 구성한 부분표 |
| `co2.index` | `Index` | pandas가 관리하는 행 label |
| `code + year` | key 후보 | 국가–연도 관측의 고유성 |

Index는 pandas 객체의 행 위치를 관리하고, `key`를 통해 관측 단위를 확정 가능
 
`code + year`의 고유성은 해당 열 조합에 대한 중복검사로 판단


In [ ]:
total = co2["emissions_total"]
identity = co2[["entity", "code", "year"]]

print('co2["emissions_total"]:', type(total).__name__, total.shape)
print('co2[["entity", "code", "year"]]:', type(identity).__name__, identity.shape)
print("pandas index preview:", co2.index[:5].tolist())


### Index와 `key` 상이

| Index | `key` |
|---|---|
| pandas가 행을 찾아가는 label | 현실의 관측을 식별하는 변수 |
| CSV를 읽을 때 0부터 자동 생성 가능 | 자료 의미를 바탕으로 연구자가 선택 |
| 현재 표 안에서 고유할 수 있음 | `code + year` 중복검사로 확인 |



## 4. 자료형·결측·중복 점검

| 검사 | 코드가 확인하는 조건 | 별도로 검토할 내용 |
|---|---|---|
| `dtypes` | 저장된 값의 형식 | 변수의 과학적 의미와 단위 |
| `isna().sum()` | 빈 셀의 수 | 조사대상·기간의 누락 |
| `duplicated(key)` | 동일 key의 반복 | key 정의의 적절성 |
| 관계식 검사 | 알려진 열 관계의 일치 | 관계식 외의 품질 문제 |

검사 결과는 대상과 조건을 붙여 기록합니다. `missing = 0`과 `duplicated code + year = 0`은 서로 다른 품질 조건을 나타냅니다.



In [ ]:
co2_audit = pd.DataFrame({
    "dtype": co2.dtypes.astype(str),
    "missing": co2.isna().sum(),
    "unique": co2.nunique(),
})
display(co2_audit)

key_audit = pd.Series({
    "all missing cells": int(co2.isna().sum().sum()),
    "duplicated full rows": int(co2.duplicated().sum()),
    "duplicated code + year": int(co2.duplicated(["code", "year"]).sum()),
}, name="count")
display(key_audit.to_frame())


### ‘0건’이 확인한 것과 확인하지 않은 것

- 결측 0건: 현재 다섯 열에 빈 셀이 발견되지 않았습니다.
- `code + year` 중복 0건: 선택한 key 후보가 파일 안에서 고유합니다.
- 관계식 검사가 통과하면 알려진 열 관계가 현재 값에서 맞습니다.

이 결과는 측정오차, 정의 변화, 누락 국가와 표본 대표성까지 평가하지 않습니다. 품질검사 결과에는 검사 범위를 함께 명시합니다.


## 5. 행 필터링과 단위 변환

한국 관측의 선택과 tonnes–Gt 변환을 구분해 실행합니다.

1. `loc`로 `code == "KOR"`인 행을 선택합니다.
2. 선택 결과의 행 수와 연도 범위를 확인합니다.
3. 총배출량을 `1 Gt = 10⁹ tonnes` 기준으로 환산합니다.

$$
\text{emissions\_gt}=\frac{\text{emissions\_total}}{10^9}
$$

필터링은 분석 대상 행을 바꾸고, 단위 변환은 같은 물리량의 표시 척도를 바꿉니다. 결과표에는 값과 단위를 나란히 표기합니다.


In [ ]:
korea = (
    co2.loc[
        co2["code"].eq("KOR"),
        ["entity", "code", "year", "emissions_total", "emissions_total_per_capita"],
    ]
    .sort_values("year")
    .assign(emissions_gt=lambda frame: frame["emissions_total"] / 1_000_000_000)
)

display(korea.head(3))
display(korea.tail(3))
print("Korea rows:", len(korea))
print(
    "unit conversion check:",
    (korea["emissions_gt"] * 1_000_000_000).round().astype("int64").eq(korea["emissions_total"]).all(),
)


### 선택 전후와 단위 변환

한국 행만 남았는지와 같은 배출량이 두 단위로 어떻게 표시되는지 나란히 봅니다.


In [ ]:
unit_view = korea.loc[
    korea["year"].isin([korea["year"].min(), korea["year"].max()]),
    ["entity", "year", "emissions_total", "emissions_gt"],
].copy()
print("전체 지역 수:", co2["code"].nunique(), "→ 선택 후 지역 수:", korea["code"].nunique())
print("tonnes와 Gt는 같은 양을 다른 단위로 표현합니다.")
display(unit_view)


### 숫자의 변화와 물리량의 변화를 구분

$$1\;\text{Gt}=10^9\;\text{tonnes}$$

같은 배출량을 Gt로 표시하면 숫자는 10억분의 1이 되지만 실제 CO₂ 양은 변하지 않습니다.

선택 전후 표에서 `emissions_total`과 `emissions_gt`를 나란히 보고 두 값의 비율을 확인합니다.


## 6. 국가와 집계지역의 구분

공개 환경자료에는 국가 외에도 `World`, 대륙, 소득집단 등의 집계지역이 포함될 수 있습니다.

| 분석 목적 | `World` 처리 |
|---|---|
| 전 세계 총량 추세 | 분석 대상에 포함 가능 |
| 국가 순위·국가 수·국가 평균 | 국가 표본에서 제외 |

`World`는 여러 국가를 결합한 관측이므로 국가 한 곳과 같은 단위로 취급할 수 없습니다. 포함·제외 기준은 연구질문에 명시합니다.


In [ ]:
scope_check = (
    co2.groupby(["entity", "code"], as_index=False)
       .agg(first_year=("year", "min"), last_year=("year", "max"), rows=("year", "size"))
       .sort_values("entity")
)
display(scope_check)

print("World is a country:", False)
print("total and per-capita have the same unit:", False)


### `World`

`World` 행은 여러 국가의 값을 합한 집계지역입니다.

- 전 세계 총량 추세를 볼 때는 유용합니다.
- 국가 순위·국가 수·국가 평균에는 자동으로 포함하지 않습니다.
- 포함하거나 제외한 기준을 연구 기록에 명시합니다.

국가와 집계지역을 섞으면 관측 단위가 달라져 비교 결과가 왜곡될 수 있습니다.


## 7. 데이터 분석 명세

분석을 시작하기 전에 표의 구조와 해석 기준을 한 곳에 기록합니다.

| 항목 | CO₂ 자료의 명세 |
|---|---|
| 분석대상 | 지역별 연간 territorial CO₂ 배출량 |
| grain | 지역 × 연도 |
| key | `code + year` |
| 변수 역할 | 식별자·시간·총량·1인당 측정값 |
| 단위 | tonnes CO₂/year, tonnes CO₂/person/year |
| 범위 | 5개 지역, 1990~2024년 |
| 출처 | OWID 공개자료의 수업용 snapshot |
| 주의사항 | `World`는 집계지역이며 두 배출량 열의 단위가 다름 |

이 명세는 집계·분할·회귀 단계에서 관측 단위나 변수 단위가 바뀌는 지점을 추적하는 기준이 됩니다.


In [ ]:
co2_contract = pd.DataFrame({
    "item": ["research object", "grain", "key", "dimensions", "measures", "known warning"],
    "record": [
        "territorial CO2 emissions by region and year",
        "region × year",
        "code + year",
        "entity, code, year",
        "total tonnes CO2/year; tonnes CO2/person/year",
        "World is an aggregate; total and per-capita answer different questions",
    ],
})
display(co2_contract)


## ChatGPT 실시간 검토: 데이터 구조에서 빠진 질문 찾기

ChatGPT에는 분석을 대신 맡기지 않고 **현재 출력만으로 놓치기 쉬운 점검 항목**을 찾게 합니다. 아래 셀의 프롬프트에는 공개 열 이름, 자료 크기, 결측·중복 요약만 들어갑니다.

답변을 받으면 `한 행`, `식별자`, `단위`, `집계지역`을 실제 DataFrame과 공식 metadata에서 다시 확인합니다. 존재하지 않는 열이나 확인되지 않은 출처가 제시되면 채택하지 않습니다.


In [ ]:
review_prompt = f"""
다음은 공개 CO₂ 자료를 처음 읽은 결과입니다.

- 자료 크기: {co2.shape}
- 열 이름: {co2.columns.tolist()}
- 연도 범위: {co2['year'].min()}~{co2['year'].max()}
- code-year 중복: {co2.duplicated(['code', 'year']).sum()}건
- 전체 결측 셀: {co2.isna().sum().sum()}개

아래 형식으로 검토해 주세요.
1. 현재 출력으로 확인된 사실
2. 한 행, 식별자 후보, 차원변수, 측정변수와 단위
3. 추가로 확인해야 할 항목 3개와 확인 방법

제약: 존재하지 않는 열·단위·출처를 만들지 말고, 확인할 수 없는 내용은 '추가 metadata 필요'라고 표시해 주세요.
""".strip()
print("[ChatGPT에 복사할 프롬프트]\n")
print(review_prompt)


### ChatGPT 출력 해석: 질문 초안과 연구 근거를 구분하기

ChatGPT가 제안한 감사 질문은 **다음 검사를 찾는 초안**입니다. 최종 기록은 다음 근거로 확인합니다.

| 검토 대상 | 확인 근거 |
|---|---|
| 열 이름·자료형 | 실제 DataFrame 출력 |
| 단위·정의 | 원자료 설명과 metadata |
| 집계지역·기간 | 실제 고유값과 범위 출력 |
| provenance | 공식 출처와 사용 snapshot 기록 |

제안은 실제 열·단위·출처와 대조해 채택 여부를 결정하고, 제외한 경우에는 그 근거를 기록합니다.


## 8. 구조 검증 결과

마지막 셀의 `PASS`는 다음 조건의 충족 여부를 확인합니다.

### 검증한 조건

- 필수 열이 존재함
- snapshot의 행 수가 175개임
- `code + year` 중복이 없음
- 한국 선택 결과가 35행임
- tonnes–Gt 변환이 수치적으로 일치함

### 별도 검토가 필요한 항목

- 측정오차와 정의 변경
- 표본의 대표성과 누락 지역
- 연구질문에 대한 변수 선택의 타당성

검증 기록에는 `PASS`가 나타내는 통과 조건을 명시합니다.


In [ ]:
validation_checks = pd.Series({
    "expected snapshot shape": co2.shape == (175, 5),
    "unique code-year key": key_audit["duplicated code + year"] == 0,
    "Korea selected": set(korea["code"]) == {"KOR"},
    "35 annual rows": len(korea) == 35,
    "unit conversion retained": (
        (korea["emissions_gt"] * 1_000_000_000)
        .round()
        .astype("int64")
        .eq(korea["emissions_total"])
        .all()
    ),
}, name="passed")
display(validation_checks.to_frame())
assert validation_checks.all()
print("자료 구조 검증: PASS")


## 해석 범위

이번 분석에서 확인한 내용은 파일의 구조, 관측 단위, 식별자 후보, 자료형, 결측, 중복과 표시 단위입니다.

이 점검만으로 측정값의 정확성, 표본의 대표성, 배출 변화의 원인이나 정책효과를 판단할 수는 없습니다. 다음 교시부터 집계와 모형을 적용할 때에도 grain과 단위를 분석 결과와 함께 기록합니다.


## 오늘의 결론과 다음 단계

### 이번 교시의 결론

파일을 읽은 뒤에는 계산보다 먼저 한 행·식별자·열 역할·단위와 품질검사 범위를 고정합니다.

### 적용 순서

1. `01_pandas_research_data_follow.ipynb`에서 핵심 시연을 같은 조건으로 재현합니다.
2. `01_pandas_research_data_practice.ipynb`에서 지정된 데이터 또는 조건 하나만 바꿔 비교합니다.
3. 변경조건, 비교표 또는 그림, 결과문장과 현재 검증하지 못한 범위를 함께 기록합니다.

### 실습 중 확인할 질문

- `co2["emissions_total"].median()`은 표의 어느 열을 선택하고 무엇을 계산하는 코드입니까?
- 평균과 중앙값이 크게 다르다면 분포와 극단값을 어떻게 추가로 점검하겠습니까?
- 한 행의 의미와 key를 명시한 뒤, 결측·중복·단위 오류 중 어느 항목을 먼저 확인해야 합니까?

마지막 `PASS`는 Notebook에 적힌 실행조건을 확인한 결과이며 연구결론의 타당성을 대신하지 않습니다.
